# Data Prep for Neural Network

In [1]:
import polars as pl

from run_config import PATHS, RUN_MODE

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The data can be split randomly (70/15/15) or chronologically.

In [2]:
DATASETS = (
    PATHS.gold_1h_demand_hexagon,
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_4h_demand_hexagon,
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: sample
Inputs: ['GOLD_1H_DEMAND_HEXAGON.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_HEXAGON.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet']
Output directory: /Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data


In [3]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RANDOM:
        bucketed = (
            df_split
            .with_row_index("_row_id")
            .with_columns(
                (pl.col("_row_id").hash(seed=SEED) % 100).alias("_split_bucket")
            )
        )
        train = bucketed.filter(pl.col("_split_bucket") < 70)
        val = bucketed.filter(
            (pl.col("_split_bucket") >= 70) & (pl.col("_split_bucket") < 85)
        )
        test = bucketed.filter(pl.col("_split_bucket") >= 85)
        helper_columns = ["_row_id", "_split_bucket"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {"counts": counts, "paths": output_paths}

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON.parquet: {'total': 5118, 'train': 0, 'val': 0, 'test': 5118}, shares={'train': 0.0, 'val': 0.0, 'test': 1.0}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_HEXAGON_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_HEXAGON_VAL.parquet'), 'test': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_HEXAGON_TEST.parquet')}
GOLD_1H_DEMAND_CENSUS_TRACTS.parquet: {'total': 5268, 'train': 0, 'val': 0, 'test': 5268}, shares={'train': 0.0, 'val': 0.0, 'test': 1.0}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_CENSUS_TRACTS_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/sample/train_test_data/GOLD_1H_DEMAND_CENSUS_TRACTS_VAL.parquet'), 'test': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/samp

In [4]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_OVC,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2026-04-30 16:00:00,4,4,16,1.0,6.1232e-17,0.433884,-0.900969,-0.866025,-0.5,8.89,70.85,10.0,10.0,0.0,0,1,2026-04-30,0,37,7.0,3.0,7.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-04-30 16:00:00,4,4,16,1.0,6.1232e-17,0.433884,-0.900969,-0.866025,-0.5,8.89,70.85,10.0,10.0,0.0,0,1,2026-04-30,0,12,21.0,2.0,8.0,2.0,1,192,192.0,192,192,1.03,1.03,1.03,1.03,5.5,5.5,5.5,5.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,6.0,6.0,6.0,"""Credit Card"""
2026-04-30 16:00:00,4,4,16,1.0,6.1232e-17,0.433884,-0.900969,-0.866025,-0.5,8.89,70.85,10.0,10.0,0.0,0,1,2026-04-30,0,74,8.0,0.0,5.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-05-01 00:00:00,5,5,0,0.866025,-0.5,-0.433884,-0.900969,0.0,1.0,7.915,70.795,8.5,10.0,0.0,1,0,2026-05-01,0,37,7.0,3.0,7.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-05-01 00:00:00,5,5,0,0.866025,-0.5,-0.433884,-0.900969,0.0,1.0,7.915,70.795,8.5,10.0,0.0,1,0,2026-05-01,0,30,34.0,18.0,29.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-04-30 16:00:00,4,4,16,1.0,6.1232e-17,0.433884,-0.900969,-0.866025,-0.5,8.89,70.85,10.0,10.0,0.0,0,1,2026-04-30,0,49,17.0,1.0,10.0,14.0,1,2387,2387.0,2387,2387,27.91,27.91,27.91,27.91,88.53,88.53,88.53,88.53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,88.53,88.53,88.53,88.53,"""Cash"""
2026-04-30 20:00:00,4,4,20,1.0,6.1232e-17,0.433884,-0.900969,-0.866025,0.5,7.915,70.795,8.5,10.0,0.5102,1,0,2026-04-30,0,74,8.0,0.0,5.0,0.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-05-01 00:00:00,5,5,0,0.866025,-0.5,-0.433884,-0.900969,0.0,1.0,7.915,70.795,8.5,10.0,0.0,1,0,2026-05-01,0,8,626.0,65.0,175.0,36.0,6,5062,843.666667,296,1457,28.15,4.691667,0.51,11.64,90.3,15.05,5.0,30.5,15.06,2.51,0.0,6.96,0.0,0.0,0.0,0.0,17.0,2.833333,0.0,15.0,123.36,20.56,6.75,30.5,"""Credit Card"""
2026-05-01 00:00:00,5,5,0,0.866025,-0.5,-0.433884,-0.900969,0.0,1.0,7.915,70.795,8.5,10.0,0.0,1,0,2026-05-01,0,60,54.0,12.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
